## Leetcode 262. Trips and Users

In [0]:
from datetime import date

trips = [
    (1, 1, 10, 1, 'completed',            date(2013,10,1)),
    (2, 2, 11, 1, 'cancelled_by_driver',  date(2013,10,1)),
    (3, 3, 12, 6, 'completed',            date(2013,10,1)),
    (4, 4, 13, 6, 'cancelled_by_client',  date(2013,10,1)),
    (5, 1, 10, 1, 'completed',            date(2013,10,2)),
    (6, 2, 11, 6, 'completed',            date(2013,10,2)),
    (7, 3, 12, 6, 'completed',            date(2013,10,2)),
    (8, 2, 12, 12,'completed',            date(2013,10,3)),
    (9, 3, 10, 12,'completed',            date(2013,10,3)),
    (10,4, 13, 12,'cancelled_by_driver',  date(2013,10,3)),
]
users = [
    (1, 'No',  'client'),
    (2, 'Yes', 'client'),
    (3, 'No',  'client'),
    (4, 'No',  'client'),
    (10,'No',  'driver'),
    (11,'No',  'driver'),
    (12,'No',  'driver'),
    (13,'No',  'driver'),
]

Trips = spark.createDataFrame(trips, ['id','client_id','driver_id','city_id','status','request_at'])
Users = spark.createDataFrame(users, ['users_id','banned','role'])
Trips.createOrReplaceTempView("Trips")
Users.createOrReplaceTempView("Users")
print("Trips")
display(Trips)
print("Users")
display(Users)

### Spark SQL

In [0]:
spark.sql('''
with banned_trips as (
    select t.id as id from
    Trips t join Users u on u.users_id = t.client_id
    where u.banned = 'Yes'

    union

    select t.id as id from
    Trips t join Users u on u.users_id = t.driver_id
    where u.banned = 'Yes'
),

unbanned_trips as (
    select * from Trips
    where id not in (select id from banned_trips)
    and request_at between '2013-10-01' and '2013-10-03'
)

select request_at as Day,
       round(count(case when status like 'canc%' then 1 end) / count(request_at), 2) as `Cancellation Rate`
from unbanned_trips
group by request_at
order by request_at
''').show()

### Spark DataFrame API

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
# banned users only
banned_users = Users.filter(col("banned") == "Yes").select("users_id")

# trips where client is banned
banned_by_client = Trips.join(banned_users, Trips.client_id == banned_users.users_id, "left_semi")

# trips where driver is banned
banned_by_driver = Trips.join(banned_users, Trips.driver_id == banned_users.users_id, "left_semi")

banned_trips = banned_by_client.union(banned_by_driver).distinct()

# now anti-join Trips against banned_trips to get only unbanned ones
unbanned_trips = Trips.join(banned_trips, on="id", how="left_anti")

# filter to the date range
unbanned_trips = unbanned_trips.filter(
    col("request_at").between("2013-10-01", "2013-10-03")
)

# compute cancellation rate per day
result = (
    unbanned_trips
    .groupBy("request_at")
    .agg(
        round(
            count(when(col("status").like("canc%"), 1)) / count("request_at"),
            2
        ).alias("Cancellation Rate")
    )
    .withColumnRenamed("request_at", "Day")
    .orderBy("Day")
)

result.show()


### 